<a href="https://colab.research.google.com/github/rdelhibabu/Spatio-Temporal_GAN/blob/main/Spatio_Temporal_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install requirements and import libraries
!pip install torch torchvision torchaudio
!pip install torch_geometric

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data
import numpy as np
import math

# Ensure reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.1 MB/s eta 0:00:00
Using device: cpu


In [2]:
# Cell 2: Data processing and k-NN Graph Construction
class SmallholderYieldDataset(torch.utils.data.Dataset):
    def __init__(self, domain="argentina", split="train"):
        """
        Mock implementation of the SustainBench data loader mirroring Section 3 of the paper.
        In production, this loads the actual .npy arrays.
        """
        self.domain = domain
        # Shapes based on the paper's Permutation-Invariant Histogram Extraction
        self.num_samples = 1620 if domain == "argentina" else 384

        # Histograms: [Batch, Time (32), Bins (32), Bands (9)] - Permuted to [B, Channels, H, W] for PyTorch 2D Conv
        self.histograms = torch.randn(self.num_samples, 9, 32, 32)

        # 1D NDVI Sequence: [Batch, Time (32)]
        self.ndvi = torch.randn(self.num_samples, 32)

        # One-hot Year: Assuming 12 years (2005-2016)
        self.years = torch.eye(12)[torch.randint(0, 12, (self.num_samples,))]

        # Target Yields: Log transformed y'_i = ln(1 + y_i)
        raw_yields = torch.abs(torch.randn(self.num_samples)) * 3
        self.yields = torch.log(1 + raw_yields)

        # Geographic Centroids (Lat/Lon) for Haversine distance
        self.centroids = torch.randn(self.num_samples, 2)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.histograms[idx], self.ndvi[idx], self.years[idx], self.yields[idx], self.centroids[idx]

def build_knn_graph(centroids, k=12):
    """
    Constructs the topological graph using k-Nearest Neighbors (Section 4.2).
    Calculates Haversine distance for edge attributes.
    """
    from sklearn.neighbors import NearestNeighbors

    # Convert lat/lon to radians for Haversine
    centroids_rad = np.radians(centroids.numpy())
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='ball_tree', metric='haversine').fit(centroids_rad)
    distances, indices = nbrs.kneighbors(centroids_rad)

    edge_index = []
    edge_attr = []

    # Calculate median distance for Gaussian Kernel Bandwidth (sigma)
    sigma = np.median(distances[:, 1:])

    for i in range(len(centroids)):
        for j in range(1, k+1): # Skip self (index 0)
            neighbor_idx = indices[i][j]
            dist = distances[i][j]

            # Eq 4: Gaussian kernel spatial weight
            weight = math.exp(-(dist**2) / (2 * sigma**2))

            edge_index.append([i, neighbor_idx])
            edge_attr.append([weight])

    return torch.tensor(edge_index, dtype=torch.long).t().contiguous(), torch.tensor(edge_attr, dtype=torch.float)

In [3]:
# Cell 3: Neural Network Architecture
class SpectralTemporalEncoder(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        # CNN for 32x32x9 Histograms
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels=9, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 64)
        )

        # NDVI MLP
        self.ndvi_mlp = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU()
        )

        # Fusion projection to get h_local
        self.fusion = nn.Linear(64 + 32 + 12, hidden_dim) # 12 is year one-hot dim

    def forward(self, histograms, ndvi, years):
        cnn_feat = self.cnn(histograms)
        ndvi_feat = self.ndvi_mlp(ndvi)
        # Concatenate features
        concat_feat = torch.cat([cnn_feat, ndvi_feat, years], dim=1)
        h_local = self.fusion(concat_feat)
        return h_local

class DifferentialSTGAT(nn.Module):
    def __init__(self, hidden_dim=128, heads=4):
        super().__init__()
        self.encoder = SpectralTemporalEncoder(hidden_dim)

        # GATv2 for Regional Aggregation (Eq 5)
        self.gat = GATv2Conv(in_channels=hidden_dim, out_channels=hidden_dim // heads,
                             heads=heads, edge_dim=1, concat=True)

        # Differential Spatial Gate components
        self.gate_linear = nn.Linear(hidden_dim, hidden_dim)
        self.local_proj = nn.Linear(hidden_dim, hidden_dim)

        # Learnable Alpha scalar initialized to 0.05 (Eq 8 & Algorithm 4)
        self.alpha = nn.Parameter(torch.tensor(0.05))

        # Final Yield Predictor
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, histograms, ndvi, years, edge_index, edge_attr):
        # 1. Local Spectral-Temporal Encoding
        h_local = self.encoder(histograms, ndvi, years)

        # 2. Regional Aggregation
        h_regional = self.gat(h_local, edge_index, edge_attr)

        # 3. Feature-wise Gate (Eq 6)
        gate = torch.sigmoid(self.gate_linear(h_regional))
        h_gated = gate * h_regional

        # 4. Compute Differential Signal (Eq 7)
        h_local_proj = self.local_proj(h_local)
        delta_h = h_local_proj - h_gated

        # 5. Combine Representations (Eq 8)
        h_combined = h_local_proj + (self.alpha * delta_h)

        # 6. Predict Yield
        prediction = self.predictor(h_combined)
        return prediction.squeeze()

In [ ]:
# Cell 4: Phased Optimization Logic
def train_phase(model, data, edge_index, edge_attr, targets, optimizer, criterion, epochs, phase_name):
    model.train()
    print(f"--- Starting {phase_name} ---")
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(data['hist'], data['ndvi'], data['years'], edge_index, edge_attr)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Alpha: {model.alpha.item():.4f}")

# Instantiate datasets and graph
source_ds = SmallholderYieldDataset(domain="argentina")
target_ds = SmallholderYieldDataset(domain="brazil")

# Extract full batch for Graph Convolution (Node Classification style)
src_hist, src_ndvi, src_years, src_y, src_coords = source_ds[:]
tgt_hist, tgt_ndvi, tgt_years, tgt_y, tgt_coords = target_ds[:]

src_edge_idx, src_edge_attr = build_knn_graph(src_coords)
tgt_edge_idx, tgt_edge_attr = build_knn_graph(tgt_coords)

model = DifferentialSTGAT().to(device)
criterion = nn.MSELoss()

# --- PHASE 1: Source Domain Pre-training ---
opt_phase1 = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
src_data = {'hist': src_hist.to(device), 'ndvi': src_ndvi.to(device), 'years': src_years.to(device)}

# CORRECTED: Added 'criterion' before '50'
train_phase(model, src_data, src_edge_idx.to(device), src_edge_attr.to(device), src_y.to(device), opt_phase1, criterion, 50, "Phase 1: Argentina Pre-training")

# --- PHASE 2: Target Domain Spatial Stabilization ---
# Freeze CNN Backbone and Alpha
for param in model.encoder.parameters():
    param.requires_grad = False
model.alpha.requires_grad = False
model.alpha.data = torch.tensor(0.05).to(device)

opt_phase2 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
tgt_data = {'hist': tgt_hist.to(device), 'ndvi': tgt_ndvi.to(device), 'years': tgt_years.to(device)}

# CORRECTED: Added 'criterion' before '40'
train_phase(model, tgt_data, tgt_edge_idx.to(device), tgt_edge_attr.to(device), tgt_y.to(device), opt_phase2, criterion, 40, "Phase 2: Brazil Spatial Stabilization")

# --- PHASE 3: Target Domain Full Fine-Tuning ---
# Unfreeze CNN and Alpha
for param in model.encoder.parameters():
    param.requires_grad = True
model.alpha.requires_grad = True

opt_phase3 = torch.optim.Adam(model.parameters(), lr=1e-4)

# CORRECTED: Added 'criterion' before '160'
train_phase(model, tgt_data, tgt_edge_idx.to(device), tgt_edge_attr.to(device), tgt_y.to(device), opt_phase3, criterion, 160, "Phase 3: Brazil Full Fine-Tuning")

--- Starting Phase 1: Argentina Pre-training ---
Epoch 0 | Loss: 1.5073 | Alpha: 0.0510
Epoch 10 | Loss: 0.2905 | Alpha: 0.0490
